# **05 · Comparación de versiones de YOLO y test con fotos públicas**

Responde a dos sugerencias de los profesores:

1. **Probar otras versiones de YOLO**, no solo `yolo26n`.
2. **Probar también con fotos del dataset público** en el test final.

**Cómo se hace una comparación justa.** Todas las versiones se entrenan con *exactamente* los mismos datos, la misma
semilla (`seed=0`), el mismo optimizador (AdamW) y los **mismos hiperparámetros que Optuna encontró para `yolo26n`**
(`results/mejores_hiperparametros.json`). Lo único que cambia es la versión del modelo.

**Por qué hay que reentrenar `yolo26n`.** En los notebooks 02 y 03 el entrenamiento usa las 915 fotos públicas
completas, así que evaluar ese modelo con fotos públicas sería evaluar con fotos que ya vio. Acá se **apartan 100 fotos
públicas** que ningún modelo de este notebook ve al entrenar. El modelo de los notebooks 02 y 03 no se toca.

**Límite conocido.** Los hiperparámetros se afinaron para `yolo26n`; las otras versiones podrían rendir algo mejor con
valores propios. Optimizar cada una por separado queda como trabajo futuro.

## **1. Preparación del entorno**

In [ ]:
!pip -q install "ultralytics==8.4.153"

In [ ]:
# Carpetas del proyecto, igual que en los notebooks anteriores (en Colab todo queda en Drive, carpeta limon_ia).
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/limon_ia")
    TRABAJO = Path("/content")
except ImportError:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    TRABAJO = BASE / "data"

DATA, RESULTS, RUNS = BASE / "data", BASE / "results", BASE / "runs"
for carpeta in (DATA, RESULTS, RUNS):
    carpeta.mkdir(parents=True, exist_ok=True)
DATOS = TRABAJO / "datos"
print("Proyecto en", BASE)

In [ ]:
import collections
import json
import random
import shutil
import time

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from ultralytics import YOLO

IMGSZ = 640
EPOCAS, PACIENCIA = 60, 20       # igual que el modelo base y el optimizado
SEMILLA = 0
N_APARTADAS = 100                # fotos públicas que no se usan para entrenar: son el test público
FRACCION = 1.0                   # 1.0 = todo el entrenamiento (solo se baja para probar el código)

# (nombre de la corrida, pesos preentrenados en COCO que descarga Ultralytics)
VERSIONES = [("yolo26n_nuevo", "yolo26n.pt"), ("yolo11n", "yolo11n.pt"), ("yolo26s", "yolo26s.pt")]

# Hiperparámetros que Optuna encontró para yolo26n (notebook 03).
ARCHIVO_HP = RESULTS / "mejores_hiperparametros.json"
if ARCHIVO_HP.exists():
    HP = json.loads(ARCHIVO_HP.read_text())
else:
    HP = {"lr0": 0.00045763170336470145, "weight_decay": 0.0001305362391823311,
          "hsv_h": 0.014675127325157949, "scale": 0.8540807039239279, "repetir": 5}
print("Hiperparámetros:", HP)

## **2. Apartar fotos públicas para el test**

Las imágenes del dataset público vienen de Roboflow: de una misma foto original pueden salir varias imágenes
(el nombre es `<foto>_jpg.rf.<código>.jpg`; lo que está antes de `.rf.` identifica la foto original). Si apartáramos
una imagen y dejáramos otra versión de la misma foto en el entrenamiento, el test estaría contaminado.

Por eso se sortean **100 fotos originales** (semilla 0), se deja **una** imagen de cada una para el test y se
**excluyen del entrenamiento todas** las imágenes de esas fotos. La lista queda en `results/fotos_apartadas.csv`.

In [ ]:
# --------
# Función que separa el dataset público en dos carpetas:
#   - base_apartadas: una imagen de cada una de las N fotos sorteadas (test público, nunca se entrena con ellas).
#   - base_resto: las imágenes de todas las demás fotos (se usan para entrenar).
#   Las imágenes de una foto sorteada que no se eligieron para el test no se usan en ningún lado.
# --------
def apartar_fotos_publicas():
    if not (DATOS / "propio").exists():
        shutil.unpack_archive(DATA / "datos_limon.zip", DATOS)
    destino_test, destino_resto = DATOS / "base_apartadas", DATOS / "base_resto"
    if destino_test.exists() and destino_resto.exists():
        return
    for d in (destino_test, destino_resto):
        shutil.rmtree(d, ignore_errors=True)
        for sub in ("images", "labels"):
            (d / sub).mkdir(parents=True)

    orden = {"test": 0, "valid": 1, "train": 2}     # para el test se prefiere una imagen original (sin aumento)
    grupos = collections.defaultdict(list)
    for conjunto in orden:
        for img in sorted((DATOS / "base" / conjunto / "images").glob("*")):
            grupos[img.name.split(".rf.")[0]].append((orden[conjunto], img.name, conjunto, img))
    sorteadas = sorted(random.Random(SEMILLA).sample(sorted(grupos), N_APARTADAS))

    def copiar(conjunto, img, destino):
        etiqueta = DATOS / "base" / conjunto / "labels" / (img.stem + ".txt")
        nuevo = f"{conjunto}_{img.name}"
        shutil.copy(img, destino / "images" / nuevo)
        if etiqueta.exists():
            shutil.copy(etiqueta, destino / "labels" / f"{conjunto}_{img.stem}.txt")

    filas = []
    for foto, imagenes in grupos.items():
        imagenes.sort(key=lambda t: (t[0], t[1]))
        if foto in sorteadas:
            _, nombre, conjunto, img = imagenes[0]
            copiar(conjunto, img, destino_test)
            filas.append({"foto_original": foto, "imagen_test": nombre, "imagenes_excluidas_del_entrenamiento": len(imagenes)})
        else:
            for _, _, conjunto, img in imagenes:
                copiar(conjunto, img, destino_resto)
    pd.DataFrame(filas).sort_values("foto_original").to_csv(RESULTS / "fotos_apartadas.csv", index=False)


apartar_fotos_publicas()

# Comprobación: ninguna foto original del test aparece en el entrenamiento.
prefijo = lambda carpeta: {p.name.split(".rf.")[0].split("_", 1)[1] for p in (DATOS / carpeta / "images").glob("*")}
assert not (prefijo("base_apartadas") & prefijo("base_resto")), "hay fotos del test en el entrenamiento"
n_test = len(list((DATOS / "base_apartadas" / "images").glob("*")))
n_lim = sum(len(p.read_text().split("\n")) - (p.read_text().endswith("\n")) for p in (DATOS / "base_apartadas" / "labels").glob("*.txt") if p.read_text().strip())
n_resto = len(list((DATOS / "base_resto" / "images").glob("*")))
print(f"Test público: {n_test} fotos con {n_lim} limones | Entrenamiento público: {n_resto} imágenes (de 915)")

## **3. Datos de entrenamiento y funciones**

Entrenamiento = fotos públicas que quedaron + las 30 fotos propias de entrenamiento repetidas `repetir` veces (igual
que en el notebook 03). Validación = las 10 fotos propias de validación (igual que en los notebooks 02 y 03).

In [ ]:
# --------
# Función que deja listo el dataset para entrenar y devuelve la ruta del .yaml.
# --------
def preparar_datos(repetir):
    rep = DATOS / "propio" / f"train_x{repetir}"
    if not rep.exists():
        for sub in ("images", "labels"):
            (rep / sub).mkdir(parents=True)
        for img in sorted((DATOS / "propio" / "train" / "images").iterdir()):
            etiqueta = DATOS / "propio" / "train" / "labels" / (img.stem + ".txt")
            for k in range(repetir):
                shutil.copy(img, rep / "images" / f"r{k}_{img.name}")
                shutil.copy(etiqueta, rep / "labels" / f"r{k}_{img.stem}.txt")
    ruta = TRABAJO / f"comparacion_x{repetir}.yaml"
    ruta.write_text(yaml.safe_dump({
        "train": [str(DATOS / "base_resto" / "images"), str(rep / "images")],
        "val": str(DATOS / "propio" / "valid" / "images"),
        "nc": 1, "names": {0: "limon"}}))
    return str(ruta)


# --------
# Función que entrena una versión con los hiperparámetros de Optuna.
#   - Si ya existe su best.pt en Drive, no vuelve a entrenar (así se puede retomar si Colab se corta).
# --------
def entrenar_version(nombre, pesos):
    mejor = RUNS / "comparacion" / nombre / "weights" / "best.pt"
    if mejor.exists():
        print(f"{nombre}: ya entrenado ({mejor})")
        return mejor
    t0 = time.time()
    YOLO(pesos).train(
        data=preparar_datos(HP["repetir"]), epochs=EPOCAS, patience=PACIENCIA, imgsz=IMGSZ, batch=16, seed=SEMILLA,
        optimizer="AdamW", lr0=HP["lr0"], weight_decay=HP["weight_decay"], hsv_h=HP["hsv_h"], scale=HP["scale"],
        fraction=FRACCION, project=str(RUNS / "comparacion"), name=nombre, exist_ok=True, plots=False, verbose=False)
    print(f"{nombre}: entrenado en {(time.time() - t0) / 60:.1f} minutos")
    return mejor


# --------
# Función que evalúa un modelo sobre una carpeta de imágenes (con sus etiquetas en la carpeta "labels" de al lado).
#   - Devuelve: precisión, recobrado, F1, mAP50 y mAP50-95. Se usa batch=16 en todas las evaluaciones.
# --------
def evaluar(modelo, carpeta_imagenes):
    ruta = TRABAJO / "eval_tmp.yaml"
    ruta.write_text(yaml.safe_dump({"train": str(carpeta_imagenes), "val": str(carpeta_imagenes),
                                    "nc": 1, "names": {0: "limon"}}))
    m = modelo.val(data=str(ruta), imgsz=IMGSZ, batch=16, project=str(TRABAJO / "evals"), name="tmp",
                   exist_ok=True, verbose=False, plots=False)
    p, r = float(m.box.mp), float(m.box.mr)
    return {"precisión": round(p, 3), "recobrado": round(r, 3), "F1": round(2 * p * r / (p + r + 1e-9), 3),
            "mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3)}


# --------
# Función que mide la velocidad: milisegundos por imagen (promedio de n imágenes, sin contar las 3 primeras).
#   - En la GPU de Colab y en la CPU de Colab. Sirve para comparar los modelos entre sí;
#     NO es la velocidad de la Raspberry Pi, que hay que medir en el robot.
# --------
def velocidad(ruta_pesos, carpeta_imagenes, n=30):
    imgs = sorted(carpeta_imagenes.glob("*"))[:n + 3]
    tiempos = {}
    for dispositivo in ("cuda:0", "cpu"):
        try:
            modelo = YOLO(str(ruta_pesos))
            ms = [modelo.predict(str(i), imgsz=IMGSZ, device=dispositivo, verbose=False)[0].speed["inference"]
                  for i in imgs][3:]
            tiempos[dispositivo] = round(sum(ms) / len(ms), 1)
        except Exception:
            tiempos[dispositivo] = None
    return {"ms_por_imagen_GPU": tiempos["cuda:0"], "ms_por_imagen_CPU": tiempos["cpu"]}

## **4. Entrenamiento**

Un modelo por celda: cada uno queda guardado en Drive apenas termina, así que si Colab se corta no se pierde lo anterior
y al volver a correr la celda se salta lo que ya está hecho. Tiempos de referencia en la T4: `yolo26n` unos 20 minutos;
`yolo11n` parecido; `yolo26s` más (entre 30 y 60 minutos, estimado).

**Orden de prioridad si falta tiempo:** `yolo26n_nuevo` (es la referencia de la comparación), luego `yolo11n`, y por
último `yolo26s`.

In [ ]:
entrenar_version(*VERSIONES[0])   # yolo26n con los datos nuevos: es la referencia de la comparación

In [ ]:
entrenar_version(*VERSIONES[1])   # yolo11n

In [ ]:
entrenar_version(*VERSIONES[2])   # yolo26s (el más lento: correrlo al final)

## **5. Evaluación**

Cada modelo se evalúa en dos conjuntos que **ninguno vio al entrenar**:

- **Propio:** las 10 fotos propias de test (58 limones).
- **Público:** las 100 fotos públicas apartadas.

Además se agrega el modelo optimizado de los notebooks 02 y 03 (si está en Drive) **solo en el test propio**: ese modelo
sí vio las fotos públicas al entrenar, así que no se lo evalúa con ellas.

In [ ]:
filas = []
test_propio, test_publico = DATOS / "propio" / "test" / "images", DATOS / "base_apartadas" / "images"

# Modelo de los notebooks 02 y 03 (entrenado con las 915 fotos públicas): solo se evalúa en las fotos propias.
anterior = RUNS / "optimizado" / "weights" / "best.pt"
if anterior.exists():
    modelo = YOLO(str(anterior))
    filas.append({"modelo": "yolo26n (notebook 03)", "conjunto": "propio", **evaluar(modelo, test_propio),
                  **velocidad(anterior, test_propio)})

for nombre, _ in VERSIONES:
    pesos = RUNS / "comparacion" / nombre / "weights" / "best.pt"
    if not pesos.exists():
        print(f"{nombre}: todavía no está entrenado, se omite")
        continue
    modelo = YOLO(str(pesos))
    try:
        parametros = sum(p.numel() for p in modelo.model.parameters()) / 1e6
    except Exception:
        parametros = None
    extra = {"parámetros_millones": round(parametros, 2) if parametros else None,
             "tamaño_MB": round(pesos.stat().st_size / 1e6, 1), **velocidad(pesos, test_publico)}
    for conjunto, carpeta in (("propio", test_propio), ("público", test_publico)):
        filas.append({"modelo": nombre, "conjunto": conjunto, **evaluar(modelo, carpeta), **extra})

comparacion = pd.DataFrame(filas)
comparacion.to_csv(RESULTS / "comparacion_versiones.csv", index=False)
comparacion

## **6. Gráfico**

In [ ]:
tabla = comparacion[comparacion["conjunto"].isin(["propio", "público"])].pivot(index="modelo", columns="conjunto", values="mAP50-95")
orden = [m for m in ["yolo26n (notebook 03)", "yolo26n_nuevo", "yolo11n", "yolo26s"] if m in tabla.index]
tabla = tabla.loc[orden]
ax = tabla.plot(kind="bar", figsize=(8, 4), rot=15)
ax.set_ylabel("mAP50-95"); ax.set_title("mAP50-95 por modelo y conjunto de test")
for contenedor, columna in zip(ax.containers, tabla.columns):   # sin etiqueta donde no hay medición
    ax.bar_label(contenedor, labels=["" if v != v else f"{v:.3f}" for v in tabla[columna]], fontsize=8)
ax.legend(title="conjunto", loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.savefig(RESULTS / "fig_comparacion_versiones.png", dpi=110)
plt.show()

## **7. Cómo interpretar los resultados**

- **Una sola semilla.** Si dos modelos difieren menos de 0,01 en mAP50-95, hay que hablar de resultados *similares*:
  otra semilla podría cambiar el orden.
- **El test propio es chico** (10 fotos, 58 limones): uno o dos limones mueven el recobrado varias centésimas.
  El test público (100 fotos) da una medida más estable.
- **No se puede dar por hecho que el test público sea más fácil.** En una verificación previa, el modelo optimizado de los
  notebooks 02 y 03, que *sí vio* estas fotos al entrenar, obtuvo solo mAP50-95 ≈ 0,52 y recobrado 0,75 sobre ellas
  (contra 0,73 y 0,95 en las fotos propias de test). Las fotos y las etiquetas públicas (polígonos convertidos a cajas)
  son distintas de las propias, así que los números públicos pueden salir más bajos. Comparar los modelos entre sí es
  válido; comparar el test público con el propio, no.
- **Los hiperparámetros están afinados para `yolo26n`.** Si este queda igual o mejor, puede deberse en parte a eso.
- **Velocidad:** los milisegundos son de Colab (GPU y CPU) y sirven para comparar los modelos entre sí. La velocidad en la
  Raspberry Pi 5 hay que medirla en el robot.